# Namenslisten gegen die Graphen prüfen

Wie gut deckt eine externe Namensliste einen Wissensgraphen ab? Zwei Zahlen,
die Verschiedenes messen und nicht ineinander umrechenbar sind:

| | misst | Konsequenz |
|---|---|---|
| **`titles_matched`** | Anteil der *Einträge* mit Entsprechung im Graphen | **Kosten**: `1/titles_matched` Ziehungen je Treffer |
| **`nodes_covered`** | Anteil der *Knoten*, den die Liste erreicht (bezogen auf \|V\|) | **Gültigkeit**: mehr kann eine Ziehung daraus nie sehen |

Die Trefferquote lässt sich durch mehr Ziehungen erkaufen, die Abdeckung nicht.
Für den Zufallssprung von DURW ist deshalb die zweite die entscheidende — sie
ist die Obergrenze dessen, was ein so emulierter Sprung erreichen kann.

Läuft aus `Code/`. Unterbau: `namelists.py` (Quellen und Auswertung),
`title_overlap.py` (Normalisierung), `build_name_index.py` (Ziehungsindizes).

In [1]:
import importlib

import pandas as pd

import config
import namelists
import title_overlap
importlib.reload(title_overlap); importlib.reload(namelists)

from graphs import loader
from namelists import NameList, SOURCES, head, name_lookup, overlap
from title_overlap import VARIANTS, Normalizer

pd.set_option("display.width", 200)

for k, v in SOURCES.items():
    flag = "realizable" if v.realizable else "COMPARISON (aus dem Graphen)"
    print(f"{k:16s} {v.path.name:44s} {flag}")

dewiki           dewiki-20260801-all-titles-in-ns0.txt        realizable
enwiki           enwiki-latest-all-titles-in-ns0.txt          realizable
top-q            top-q-entities                               realizable
indeg-gpt4_io    gpt4_io__in-degree.txt                       COMPARISON (aus dem Graphen)
indeg-gpt4o_io   gpt4o_io__in-degree.txt                      COMPARISON (aus dem Graphen)


## Der Baukasten

`lookup` ist die teure Struktur: ein dict über alle normalisierten Knotennamen,
bei 6,5 Mio Knoten rund 26 s und ein bis zwei GB. Er hängt nur am **Normalizer**,
nicht an der Liste — deshalb einmal je (Graph, Normalizer) bauen und an alle
`overlap()`-Aufrufe weiterreichen. Jeder weitere Anteil kostet dann fast nichts.

In [2]:
def evaluate(graph_name, jobs, log=True):
    """jobs: Liste von (Beschriftung, NameList) -> DataFrame.

    Gruppiert nach Normalizer, damit der lookup je Normalizer nur einmal
    entsteht."""
    loader.clear_cache()
    g = loader.load_graph(graph_name)
    rows, lookups = [], {}
    for label, nl in jobs:
        if nl.norm not in lookups:
            if log: print(f"  lookup für {nl.norm.label()} ...", flush=True)
            lookups[nl.norm] = name_lookup(g, nl.norm)
        r = overlap(g, nl, lookup=lookups[nl.norm])
        rows.append({"liste": label, "Einträge": r["n_entries"],
                     "Treffer %": round(r["titles_matched"] * 100, 2),
                     "Abdeckung %": round(r["nodes_covered"] * 100, 3),
                     "Knoten": r["n_nodes_hit"],
                     "Zieh./Treffer": round(r["draws_per_hit"], 1)})
        if log: print(f"  {label:28s} Treffer {r['titles_matched']:7.2%}  "
                      f"Abdeckung {r['nodes_covered']:7.3%}", flush=True)
    del lookups
    return pd.DataFrame(rows).set_index("liste")

## Anteile beliebig festlegen

`head(quelle, fraction=…)` bzw. `head(quelle, n=…)` schneidet eine **sortierte**
Liste vorne ab: `top-q` ist nach QRank geordnet, die In-Grad-Listen nach
Eingangsgrad. Bei den alphabetischen Wikipedia-Dumps warnt die Funktion, weil ein
Präfix dort keine „Top-n" wäre, sondern alles, was mit `!` und `A` anfängt.

`FRACTIONS` ist der einzige Knopf.

In [3]:
FRACTIONS = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]

def fraction_jobs(source, fractions=FRACTIONS):
    return [(f"{source} {f:>6.1%}", head(source, fraction=f)) for f in fractions]

GRAPH = "gpt4_io"          # oder "gpt4o_io"

jobs = (fraction_jobs("top-q")
        + fraction_jobs("indeg-gpt4_io")      # eigene Liste = Obergrenze
        + fraction_jobs("indeg-gpt4o_io"))    # fremde Liste = echter Test

df = evaluate(GRAPH, jobs)
df

  lookup für nfc+casefold ...
  top-q   1.0%                 Treffer  91.60%  Abdeckung  0.014%
  top-q   5.0%                 Treffer  88.92%  Abdeckung  0.068%
  top-q  10.0%                 Treffer  87.22%  Abdeckung  0.134%
  top-q  25.0%                 Treffer  84.31%  Abdeckung  0.323%
  top-q  50.0%                 Treffer  81.33%  Abdeckung  0.621%
  top-q 100.0%                 Treffer  78.38%  Abdeckung  1.190%
  lookup für nfc+underscores_to_spaces+casefold+collapse_whitespace ...
  indeg-gpt4_io   1.0%         Treffer 100.00%  Abdeckung  0.996%
  indeg-gpt4_io   5.0%         Treffer 100.00%  Abdeckung  4.965%
  indeg-gpt4_io  10.0%         Treffer 100.00%  Abdeckung  9.912%
  indeg-gpt4_io  25.0%         Treffer 100.00%  Abdeckung 24.705%
  indeg-gpt4_io  50.0%         Treffer  99.99%  Abdeckung 49.350%
  indeg-gpt4_io 100.0%         Treffer  99.98%  Abdeckung 98.714%
  indeg-gpt4o_io   1.0%        Treffer  91.27%  Abdeckung  0.795%
  indeg-gpt4o_io   5.0%        Treffer  

,Einträge,Treffer %,Abdeckung %,Knoten,Zieh./Treffer
liste,,,,,
top-q 1.0%,1000,91.60,0.014,916,1.1
top-q 5.0%,5000,88.92,0.068,4434,1.1
top-q 10.0%,10000,87.22,0.134,8685,1.1
top-q 25.0%,25000,84.31,0.323,20949,1.2
top-q 50.0%,50000,81.33,0.621,40289,1.2
top-q 100.0%,100000,78.38,1.190,77241,1.3
indeg-gpt4_io 1.0%,64926,100.00,0.996,64671,1.0
indeg-gpt4_io 5.0%,324629,100.00,4.965,322376,1.0
indeg-gpt4_io 10.0%,649259,100.00,9.912,643539,1.0


## Beide Graphen nebeneinander

**Zu den In-Grad-Listen:** sie stammen aus dem Graphen selbst
(`realizable=False`). Gegen den *eigenen* Graphen ist die Trefferquote per
Konstruktion 100 % und die Abdeckung genau der genommene Anteil — das ist die
Obergrenze, kein Verfahren. Die eigentliche Frage ist der **Kreuztest**: wie weit
deckt die Prominenzliste des einen Graphen den anderen ab?

In [ ]:
both = {}
for graph in ("gpt4_io", "gpt4o_io"):
    both[graph] = evaluate(graph, jobs, log=False)
    print(f"{graph} fertig", flush=True)

pd.concat(both, names=["graph"])[["Treffer %", "Abdeckung %", "Zieh./Treffer"]]

## Vollständige Wikipedia-Dumps

Ohne Anteil, weil alphabetisch sortiert — ein Präfix wäre keine sinnvolle
Teilmenge. Rechnet einige Minuten (enwiki hat 19,2 Mio Einträge).

In [ ]:
wiki_jobs = [("enwiki", SOURCES["enwiki"]), ("dewiki", SOURCES["dewiki"])]
pd.concat({g: evaluate(g, wiki_jobs, log=False) for g in ("gpt4_io", "gpt4o_io")},
          names=["graph"])

## Eine neue Datei anschließen

Für eine Liste, die noch nicht in `SOURCES` steht, reicht ein `NameList`. Danach
in `namelists.SOURCES` eintragen, wenn sie dauerhaft bleiben soll — erst dann
entstehen Registry-Einträge, und erst dann ist ein Ziehungsindex baubar:

    python build_name_index.py --graphs gpt4_io --sources <name>

In [ ]:
# Zeilenliste, ein Name je Zeile:
eigene = NameList(config.ADDITIONALS_DIR / "meine_namen.txt",
                  VARIANTS["+whitespace"])

# Tabelle ohne Kopfzeile, Name in Spalte 1, Tab-getrennt (wie die In-Grad-Listen):
eigene_tsv = NameList(config.ADDITIONALS_DIR / "meine.tsv",
                      VARIANTS["+whitespace"],
                      columns=(1,), delimiter="\t", skip_header=False)

# Tabelle mit Kopfzeile, mehrere Namensspalten in Prioritätsreihenfolge
# (erster Treffer gewinnt, gezogen wird pro Eintrag -- nicht pro Name):
eigene_csv = NameList(config.ADDITIONALS_DIR / "meine.csv",
                      Normalizer(nfc=True, underscores_to_spaces=False,
                                 casefold=True, collapse_whitespace=False),
                      columns=("enwiki_title", "label"))

# evaluate(GRAPH, [("meine Liste", eigene)])

## Verlauf: coverage gegen surplus

Bisher: beide Größen für *eine* Listenlänge. Jetzt der **Verlauf** über die
Präfixlänge, damit ablesbar wird, wie viele Einträge sich lohnen.

| | Definition | bestimmt |
|---|---|---|
| **coverage(n)** | verschiedene getroffene Knoten in den ersten n Einträgen / \|V\| | **Gültigkeit** — mehr kann eine Ziehung nie sehen |
| **surplus(n)** | (n − Treffer(n)) / n | **Kosten** — jeder ist ein Fehlschlag beim Rejection Sampling |

Mehr Einträge heben beides. Wo der Handel kippt, ist die Frage.

**Die Zuordnung ist über Kreuz**: die In-Grad-Liste läuft nie gegen ihren
eigenen Graphen. Dort wäre die Trefferquote per Konstruktion 100 % und die
Abdeckung genau der genommene Anteil — eine Tautologie ohne Aussage. Gegen den
*anderen* Graphen ist sie dagegen ein echter Test.

| Grundmenge | Testmengen |
|---|---|
| `gpt4_io` | `top-q`, `indeg-gpt4o_io` |
| `gpt4o_io` | `top-q`, `indeg-gpt4_io` |

In [ ]:
import numpy as np

from namelists import coverage_curve
from plotting.name_coverage import plot_curve

# Log-Stützstellen von 1 bis 1 Mio, rund 48 je Dekade. Ein Checkpoint kostet
# nur zwei Zählerabfragen, die Dichte ist also praktisch gratis und macht den
# Knick gut sichtbar.
CHECKPOINTS = np.unique(np.logspace(0, 6, 300).astype(np.int64))
print(f"{len(CHECKPOINTS)} Stützstellen von {CHECKPOINTS[0]} bis {CHECKPOINTS[-1]:,}")

PAIRS = {
    "gpt4_io":  [("top-q (QRank)", SOURCES["top-q"]),
                 ("in-degree of gpt4o_io", SOURCES["indeg-gpt4o_io"])],
    "gpt4o_io": [("top-q (QRank)", SOURCES["top-q"]),
                 ("in-degree of gpt4_io", SOURCES["indeg-gpt4_io"])],
}

### Berechnung

`name_lookup` ist bei 6,5 Mio Knoten ein bis zwei GB, und die beiden Quellen
eines Graphen benutzen **verschiedene** Normalizer (`top-q` faltet nur
Groß/Klein, die In-Grad-Listen zusätzlich Unterstriche und Whitespace). Deshalb
je Graph nacheinander und jeden Lookup sofort freigeben — sonst liegen zwei
gleichzeitig im Speicher.

In [ ]:
import gc, time

curves = {}
for graph_name, sources in PAIRS.items():
    loader.clear_cache()
    t0 = time.perf_counter()
    g = loader.load_graph(graph_name)
    print(f"[{graph_name}] |V| = {g.n_nodes:,} (geladen in {time.perf_counter()-t0:.0f}s)",
          flush=True)
    curves[graph_name] = {}
    for label, src in sources:
        t0 = time.perf_counter()
        lookup = name_lookup(g, src.norm)          # 1-2 GB
        c = coverage_curve(g, src, CHECKPOINTS, lookup=lookup)
        del lookup; gc.collect()
        curves[graph_name][label] = c
        print(f"  {label:24s} bei n={c['n'][-1]:>9,}: "
              f"coverage {c['coverage'][-1]:6.2%}  surplus {c['surplus'][-1]:6.2%}"
              f"   ({time.perf_counter()-t0:.0f}s)", flush=True)

### Prüfung, bevor geplottet wird

`coverage` **muss** monoton steigen — Knoten kommen nur dazu, nie weg. Eine
Verletzung wäre ein Fehler in der Mengenführung. `surplus` muss das *nicht*: bei
einer nach Relevanz sortierten Liste steigt es erwartbar, ein Rückgang wäre aber
legitim, wenn ein späterer Block besser trifft.

Zusätzlich die Gegenprobe gegen `overlap()` an einer festen Stelle — beide Wege
müssen exakt dasselbe liefern, sonst benutzen sie verschiedene Trefferregeln.

In [ ]:
for graph_name, cs in curves.items():
    for label, c in cs.items():
        cov = np.asarray(c["coverage"])
        assert np.all(np.diff(cov) >= 0), f"{graph_name}/{label}: coverage fällt"
        sur = np.asarray(c["surplus"])
        print(f"{graph_name:9s} {label:24s} coverage monoton ✓   "
              f"surplus {sur[0]:.1%} → {sur[-1]:.1%}"
              f"{'  (nicht monoton)' if np.any(np.diff(sur) < -1e-12) else ''}")

# Gegenprobe gegen die Einzelauswertung -- zwingend an *derselben* Stelle:
# searchsorted liefert die naechste Stuetzstelle, und schon ein paar tausend
# Eintraege Unterschied verschieben beide Werte sichtbar.
loader.clear_cache()
g = loader.load_graph("gpt4_io")
c = curves["gpt4_io"]["top-q (QRank)"]
print("\nGegenprobe (top-q gegen gpt4_io):")
for target in (1_000, 100_000):
    i = int(np.searchsorted(c["n"], target))
    n = c["n"][i]                                   # exakt die Stuetzstelle
    ref = overlap(g, head(SOURCES["top-q"], n=n))
    ok = (abs(c["coverage"][i] - ref["nodes_covered"]) < 1e-12 and
          abs(c["surplus"][i] - (1 - ref["titles_matched"])) < 1e-12)
    print(f"  n = {n:>7,}  coverage {c['coverage'][i]:.8%} / {ref['nodes_covered']:.8%}"
          f"   surplus {c['surplus'][i]:.8%} / {1-ref['titles_matched']:.8%}"
          f"   -> {'identisch' if ok else 'ABWEICHUNG'}")

### Die vier Diagramme

Je Graph eines für coverage und eines für surplus, x logarithmisch bis 1 Mio.
Senkrechte Stützlinien bei 10k / 100k / 1M und waagerechte bei runden Prozenten,
damit sich beide Diagramme am selben x zusammenlesen lassen.

In [ ]:
from IPython.display import Image, display

paths = []
for graph_name, cs in curves.items():
    for metric in ("coverage", "surplus"):
        # y-Achse je Groesse: log fuer coverage (sechs Dekaden),
        # linear fuer surplus (Prozentpunkte). Default aus METRICS.
        p = plot_curve(cs, graph_name, metric)
        paths.append(p)
        print(p)

for p in paths:
    display(Image(filename=str(p)))